# 3 — One cohort, from the command line

Notebook 1 calls the library. This one types the commands, so what you read here is
what a terminal — or a Nextflow process — runs. Same clustering, same *p*-values.

Both methods appear side by side: **pvclust** and **k-means**, each with AU.

It runs on the SLE proteomics data if you have it, and on a stand-in of the same
shape if you do not, so every cell executes either way.

In [ ]:
import os, shutil, subprocess, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Everything is written here, so nothing lands in the repository.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the
    command line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd if isinstance(cmd, list) else [cmd]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

## The data

Download from Zenodo (doi:10.5281/zenodo.20342569), unpack it, and point `SLE` at
the folder. Without it the cells below still run, against a generated stand-in that
has the same awkward features: two batches, a case/control split, and a few
proteins measured by more than one reagent.

In [ ]:
# The SLE data is not in the repository -- download it from Zenodo
# (doi:10.5281/zenodo.20342569) and point SLE at the unpacked folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 100
    abundance = pd.read_csv(SLE / 'abundance.csv').set_index('SampleId')
    meta = pd.read_csv(SLE / 'sample-metadata.csv').set_index('SampleId')
    meta = meta[meta['Included_in_study'] == 'Included']
    abundance = abundance.loc[meta.index]
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
    print(f'SLE data: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')
else:
    # A stand-in with the same shape of problem, so every command below runs
    # unchanged without the download: two batches, a case/control split, and a
    # few proteins measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    rng = np.random.default_rng(0)
    n, p = 75, 30
    ids = [f'S{i:03d}' for i in range(n)]
    drivers = rng.normal(size=(n, 6))
    X = np.exp(rng.normal(3, 1, size=(1, p)) + drivers @ rng.normal(size=(6, p))
               + rng.normal(scale=0.3, size=(n, p)))
    seqs = [f'seq.{1000+j}.{j%7}' for j in range(p)]
    abundance = pd.DataFrame(X, index=pd.Index(ids, name='SampleId'), columns=seqs)
    batch = np.where(np.arange(n) % 3 == 0, 'B', 'A')
    abundance.loc[batch == 'B'] *= 1.6                     # a real batch shift
    meta = pd.DataFrame({
        'DonorId': ids, 'Batch': batch,
        'Group': np.where(rng.random(n) < 0.25, 'HV', 'SLE'),
        'Sex': rng.choice(['F', 'M'], n, p=[0.85, 0.15]),
        'Age_group': rng.choice(['26-30', '31-35', '36-40', '41-45'], n),
        'Disease_activity': rng.choice(['Remission', 'LDA', 'MDA', 'HDA'], n),
        'SLEDAI_2K': rng.integers(0, 14, n)}, index=pd.Index(ids, name='SampleId'))
    # names, with three proteins deliberately measured twice
    gene = [f'G{j:02d}' for j in range(p)]
    for a, b in [(1, 2), (10, 11), (20, 21)]:
        gene[b] = gene[a]
    fm = pd.DataFrame({'SeqId': seqs, 'TargetFullName': gene, 'GeneSymbol': gene})
    dup = fm['GeneSymbol'].duplicated(keep=False)
    fm.loc[dup, 'GeneSymbol'] = fm.loc[dup, 'GeneSymbol'] + '_' + fm.loc[dup, 'SeqId']
    fm.to_csv('feature_metadata.txt', sep='\t', index=False)
    FEATURE_MAP = 'feature_metadata.txt'
    print('SLE data not found -- using a stand-in of the same shape.')
    print(f'stand-in: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')

## Three cohorts

Split by donor rather than by sample. A donor with two timepoints must land wholly
in one cohort, or the cohorts share subjects and the federation is measuring the
same person twice.

In [ ]:
# Three cohorts, donors kept whole so repeat timepoints never straddle a boundary.
rng = np.random.default_rng(42)
donors = meta.groupby('DonorId').size().index.to_numpy()
who = dict(zip(rng.permutation(donors), range(len(donors))))
which = meta['DonorId'].map(lambda d: 'ABC'[who[d] % 3])

# SLEDAI banded, so it reads as a strip rather than fifteen shades of one colour.
out = meta.copy()
out['SLEDAI_band'] = pd.cut(pd.to_numeric(out['SLEDAI_2K'], errors='coerce'),
                            [-0.1, 0, 4, 8, 30], labels=['0', '1-4', '5-8', '9+'])
out = out.astype({'SLEDAI_band': str}).replace('nan', 'NA').fillna('NA')
out.to_csv('meta.csv')
abundance.to_csv('cohort_all.csv')
for c in 'ABC':
    abundance.loc[which[which == c].index].to_csv(f'cohort{c}.csv')
print({c: int((which == c).sum()) for c in 'ABC'})

## Pick the objects once

Every cohort has to cluster the **same** objects, or nothing pools. So the most
variable proteins are chosen once, on the pooled data, and that list is handed to
each cohort with `--shared-features`.

Note the order inside the command: log2, then ComBat, then the variance ranking.
Ranking before correcting would rank the batch shift.

In [ ]:
run(['pvclust-py', 'project-features', '--project', 'all',
     '--matrix', 'cohort_all.csv', '--log2',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--metadata', 'meta.csv', '--top-variable', str(TOPVAR),
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol'])

In [ ]:
# The flags every command shares. Written out in full each time below, so you can
# copy any single cell straight into a terminal.
COMMON = ['--log2', '--adjust', 'combat', '--batch-col', 'Batch',
          '--protect', 'Group', '--metadata', 'meta.csv',
          '--shared-features', 'all_features.csv',
          '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol']
DIST = ['--dist', 'correlation', '--linkage', 'average']
print(' '.join(COMMON))

## pvclust, one cohort at a time

`--plot` writes the dendrogram with AU in red and BP in green, and boxes the
clusters that clear `--alpha`. Each cohort is corrected against **its own** batches,
which is what a real site would do.

In [ ]:
for c in 'ABC':
    run(['pvclust-py', 'cluster', '--project', f'cohort{c}',
         '--matrix', f'cohort{c}.csv', *COMMON, *DIST,
         '--n-boot', str(NBOOT), '--alpha', '0.95', '--plot'])

In [ ]:
show('cohortA_dendrogram.png')

## k-means, with the same AU treatment

`--jaccard` is not optional here. k-means has no tree, so a cluster is matched
across bootstrap replicates by its membership, and demanding an *exact* member set
drives BP to zero as the object count grows — 0.83 at 20 objects, 0.20 at 50, and
0.00 at 150. Jaccard overlap at 0.6 is what keeps the count meaningful.

In [ ]:
for c in 'ABC':
    run(['pvclust-py', 'kmeans', '--project', f'cohort{c}', '--k', '10',
         '--matrix', f'cohort{c}.csv', *COMMON,
         '--n-boot', str(NBOOT), '--jaccard', '0.6', '--plot'])

## The two-way figure

Both axes clustered and ordered by support, with the demographics as strips down
the side. This is also the batch-effect check: if the samples cluster by plate
rather than by biology, the strips show it immediately.

`--method` chooses which clustering orders the axes.

In [ ]:
ANN = ['--annotate', 'Group,Sex,Age_group,Disease_activity,SLEDAI_band',
       '--max-rows', '400', '--max-labels', '160']
run(['pvclust-py', 'heatmap', '--project', 'cohortA',
     '--matrix', 'cohortA.csv', *COMMON, *DIST, '--method', 'pvclust',
     '--n-boot', str(NBOOT), *ANN])

In [ ]:
show('cohortA_heatmap_pvclust.png')

In [ ]:
run(['pvclust-py', 'heatmap', '--project', 'cohortA',
     '--matrix', 'cohortA.csv', *COMMON, *DIST, '--method', 'kmeans', '--k', '10',
     '--n-boot', str(NBOOT), '--jaccard', '0.6', *ANN])

In [ ]:
show('cohortA_heatmap_kmeans.png')

---
Each cohort now has its own answer. Notebook 4 pools them without any cohort
sending a single subject-level row.